In [ ]:
# ============================================================
# GRAD-CAM & GRAD-CAM++ IMPLEMENTATION
# ============================================================
import cv2
import os

# Safety: redefine custom layers + reload model if kernel was restarted
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

try:
    model
except NameError:
    model = tf.keras.models.load_model(
        "/kaggle/working/best_model.keras",
        custom_objects={"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}
    )
    print("Model reloaded from checkpoint")

# ========================== AUTO-DETECT TARGET CONV LAYER ==========================
# Last spatial (4D) feature map before GlobalAveragePooling -> CBAM refined feature.
# NOTE (Keras 3): plain layers no longer expose `.output_shape`, only nested
# Model instances (like the MobileNetV2 submodel) do -> using output_shape would
# wrongly select the raw encoder instead of the CBAM-refined feature, and
# extracting an intermediate tensor from a nested submodel breaks graph tracing
# (KeyError). So: use `.output.shape` and explicitly skip nested Model layers.
def get_last_spatial_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.Model):
            continue  # skip nested submodels (e.g. the MobileNetV2 encoder)
        try:
            out = layer.output
        except Exception:
            continue
        out_shape = out.shape if not isinstance(out, list) else out[0].shape
        if out_shape is not None and len(out_shape) == 4:
            return layer
    raise ValueError("No 4D spatial layer found in model")

TARGET_LAYER = get_last_spatial_layer(model)
print("Grad-CAM target layer:", TARGET_LAYER.name, TARGET_LAYER.output.shape)

GRAD_MODEL = tf.keras.models.Model(
    inputs=model.input,
    outputs=[TARGET_LAYER.output, model.output]
)

# ========================== GRAD-CAM ==========================
def compute_gradcam(img_array, pred_index=None):
    with tf.GradientTape() as tape:
        conv_output, preds = GRAD_MODEL(img_array, training=False)
        preds = preds[0]
        if pred_index is None:
            pred_index = int(tf.argmax(preds))
        class_channel = tf.expand_dims(preds[pred_index], axis=0)

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]

    heatmap = tf.reduce_sum(conv_output * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    confidence = float(preds[pred_index].numpy())
    return heatmap.numpy(), pred_index, confidence

# ========================== GRAD-CAM++ ==========================
def compute_gradcam_plusplus(img_array, pred_index=None):
    with tf.GradientTape() as tape:
        conv_output, preds = GRAD_MODEL(img_array, training=False)
        preds = preds[0]
        if pred_index is None:
            pred_index = int(tf.argmax(preds))
        class_channel = tf.expand_dims(preds[pred_index], axis=0)

    grads = tape.gradient(class_channel, conv_output)
    conv_output = conv_output[0]
    grads = grads[0]

    grads_sq = tf.square(grads)
    grads_cube = grads_sq * grads

    sum_A = tf.reduce_sum(conv_output, axis=(0, 1), keepdims=True)
    alpha_denom = 2.0 * grads_sq + sum_A * grads_cube
    alpha_denom = tf.where(alpha_denom != 0, alpha_denom, tf.ones_like(alpha_denom) * 1e-8)
    alphas = grads_sq / alpha_denom

    weights = tf.reduce_sum(alphas * tf.nn.relu(grads), axis=(0, 1))
    heatmap = tf.reduce_sum(conv_output * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    confidence = float(preds[pred_index].numpy())
    return heatmap.numpy(), pred_index, confidence

# ========================== OVERLAY HELPER ==========================
def overlay_heatmap(img_float, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img_float.shape[1], img_float.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = heatmap_color.astype(np.float32) * alpha + img_float
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay, heatmap_resized

# ========================== SAMPLE VISUALIZATION (ALL CLASSES) ==========================
# ager code e prothom N ta val image niche felto (sequential index) -> jodi val set
# order e shob class thake na ba ekhane 2 class dominant thake, তাহলে shudhu 1-2 class e
# dekha jeto. Ekhon protita class theke explicitly n_samples ta kore image select kora hocche.
SAMPLES_PER_CLASS = 1  # protita class theke koyta shomple dekhte chao, dorkar hole barao

val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

n_rows = len(selected_indices)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(selected_indices):
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()
    img_batch = img_np[np.newaxis, ...]

    heatmap_gc, pred_idx, conf_gc = compute_gradcam(img_batch)
    overlay_gc, _ = overlay_heatmap(img_np, heatmap_gc)

    heatmap_gcpp, _, conf_gcpp = compute_gradcam_plusplus(img_batch, pred_index=pred_idx)
    overlay_gcpp, _ = overlay_heatmap(img_np, heatmap_gcpp)

    axes[row, 0].imshow(img_np.astype(np.uint8))
    axes[row, 0].set_title(f"Original\nTrue: {class_names[true_label]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay_gc)
    axes[row, 1].set_title(f"Grad-CAM\nPred: {class_names[pred_idx]} ({conf_gc*100:.2f}%)")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay_gcpp)
    axes[row, 2].set_title(f"Grad-CAM++\nPred: {class_names[pred_idx]} ({conf_gcpp*100:.2f}%)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/gradcam_vs_gradcampp.png", dpi=600)
plt.show()
print("Saved: gradcam_vs_gradcampp.png (all", len(class_names), "classes covered)")


In [ ]:
# ============================================================
# INSERTION / DELETION AUC (Grad-CAM & Grad-CAM++)
# Patch-based causal faithfulness metric (Petsiuk et al., RISE)
# Session-resumable: safe against Kaggle kernel resets
# ============================================================
CHECKPOINT_PATH = "/kaggle/working/xai_causal_metrics_progress.csv"
SUMMARY_PATH = "/kaggle/working/xai_causal_metrics_summary.csv"

N_EVAL_SAMPLES = min(150, len(val_paths))   # bariye/komiye dite paro time budget onujayi
PATCH_SIZE = 16
N_STEPS = 20

def get_patch_order(heatmap_224, patch_size=16):
    h, w = heatmap_224.shape
    n_h, n_w = h // patch_size, w // patch_size
    patch_scores = np.zeros((n_h, n_w))
    for i in range(n_h):
        for j in range(n_w):
            patch_scores[i, j] = heatmap_224[
                i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size
            ].mean()
    flat_order = np.argsort(-patch_scores.ravel())
    order = np.stack(np.unravel_index(flat_order, patch_scores.shape), axis=1)
    return order

def causal_metric(img_224, heatmap_224, pred_index, mode="insertion",
                   patch_size=16, n_steps=20):
    order = get_patch_order(heatmap_224, patch_size)
    n_patches = len(order)
    step = max(1, n_patches // n_steps)

    blurred = cv2.GaussianBlur(img_224, (31, 31), 0)
    canvas = blurred.copy() if mode == "insertion" else img_224.copy()

    scores, fractions = [], []
    for idx in range(0, n_patches + step, step):
        idx = min(idx, n_patches)
        for (pi, pj) in order[:idx]:
            r0, r1 = pi*patch_size, (pi+1)*patch_size
            c0, c1 = pj*patch_size, (pj+1)*patch_size
            if mode == "insertion":
                canvas[r0:r1, c0:c1] = img_224[r0:r1, c0:c1]
            else:
                canvas[r0:r1, c0:c1] = blurred[r0:r1, c0:c1]

        pred = model.predict(canvas[np.newaxis, ...], verbose=0)[0]
        scores.append(pred[pred_index])
        fractions.append(idx / n_patches)
        if idx == n_patches:
            break

    return float(np.trapz(scores, fractions))

# ========================== RESUME FROM CHECKPOINT ==========================
if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)
    done_indices = set(results_df["image_index"].tolist())
    print(f"Resuming... {len(done_indices)}/{N_EVAL_SAMPLES} images already done")
else:
    results_df = pd.DataFrame(columns=[
        "image_index", "true_label", "pred_label", "confidence",
        "GradCAM_Insertion_AUC", "GradCAM_Deletion_AUC",
        "GradCAM++_Insertion_AUC", "GradCAM++_Deletion_AUC"
    ])
    done_indices = set()

eval_paths = val_paths[:N_EVAL_SAMPLES]
eval_labels = val_labels[:N_EVAL_SAMPLES]

for idx, (path, true_label) in enumerate(zip(eval_paths, eval_labels)):
    if idx in done_indices:
        continue

    img, _ = load_image(path, true_label)
    img_np = img.numpy()
    img_batch = img_np[np.newaxis, ...]

    heatmap_gc, pred_idx, confidence = compute_gradcam(img_batch)
    heatmap_gc_224 = cv2.resize(heatmap_gc, (IMG_SIZE, IMG_SIZE))

    heatmap_gcpp, _, _ = compute_gradcam_plusplus(img_batch, pred_index=pred_idx)
    heatmap_gcpp_224 = cv2.resize(heatmap_gcpp, (IMG_SIZE, IMG_SIZE))

    ins_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "deletion", PATCH_SIZE, N_STEPS)
    ins_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "deletion", PATCH_SIZE, N_STEPS)

    results_df.loc[len(results_df)] = [
        idx, true_label, pred_idx, confidence, ins_gc, del_gc, ins_gcpp, del_gcpp
    ]

    if idx % 10 == 0:
        results_df.to_csv(CHECKPOINT_PATH, index=False)
        print(f"[{idx+1}/{N_EVAL_SAMPLES}] checkpoint saved")

results_df.to_csv(CHECKPOINT_PATH, index=False)

# ========================== SUMMARY ==========================
summary = pd.DataFrame({
    "Mean_Confidence":         [results_df["confidence"].mean()],
    "GradCAM_Insertion_AUC":   [results_df["GradCAM_Insertion_AUC"].mean()],
    "GradCAM_Deletion_AUC":    [results_df["GradCAM_Deletion_AUC"].mean()],
    "GradCAM++_Insertion_AUC": [results_df["GradCAM++_Insertion_AUC"].mean()],
    "GradCAM++_Deletion_AUC":  [results_df["GradCAM++_Deletion_AUC"].mean()],
})
summary.to_csv(SUMMARY_PATH, index=False)

print("\n======================")
print("XAI Causal Faithfulness Metrics (mean over", len(results_df), "images)")
print("======================")
print(summary.T)
